In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/appledisease/Test/Blotch_Apple/149.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/143.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/131.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/130.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/120.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/124.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/123.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/137.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/127.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/140.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/145.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/135.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/121.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/126.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/142.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/148.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/138.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/146.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/1

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

2025-11-27 22:10:51.983514: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764281452.173757      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764281452.226497      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
train_dir = "/kaggle/input/appledisease/Train"
test_dir = "/kaggle/input/appledisease/Test"

In [4]:
img_size = (224, 224)
batch_size = 32
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [5]:
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
test_datagen = ImageDataGenerator(rescale=1./255)

In [6]:
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

Found 307 images belonging to 4 classes.


In [7]:
val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 75 images belonging to 4 classes.


In [8]:
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 120 images belonging to 4 classes.


In [9]:
for subdir, _, files in os.walk(train_dir):
    for file in files:
        try:
            img_path = os.path.join(subdir, file)
            img = Image.open(img_path)
            img.verify()
        except Exception as e:
            print(f"Corrupted or unreadable file: {img_path} — {e}")

In [10]:
labels_from_gen = train_gen.classes
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_from_gen),
    y=labels_from_gen
)
class_weight = dict(enumerate(class_weight))

In [11]:
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

In [12]:
print('phase1...training and freezing')
base_model = DenseNet121(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
base_model.trainable = False

phase1...training and freezing


I0000 00:00:1764281659.927043      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [13]:
model = Sequential()
model.add(base_model)
model.add(GlobalAveragePooling2D())
model.add(Dropout(0.3))
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(train_gen.num_classes, activation='softmax'))

In [14]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [15]:
trainig1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight,
    callbacks=[early_stop, lr_reduce]
)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1764281690.557633     117 service.cc:148] XLA service 0x7a12b4003870 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764281690.558557     117 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1764281693.237274     117 cuda_dnn.cc:529] Loaded cuDNN version 90300


 1/10 ━━━━━━━━━━━━━━━━━━━━ 4:34 30s/step - accuracy: 0.2105 - loss: 1.7666

I0000 00:00:1764281703.483776     117 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10/10 ━━━━━━━━━━━━━━━━━━━━ 67s 4s/step - accuracy: 0.2584 - loss: 1.7109 - val_accuracy: 0.3067 - val_loss: 1.3875 - learning_rate: 1.0000e-04
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 508ms/step - accuracy: 0.2864 - loss: 1.5693 - val_accuracy: 0.4533 - val_loss: 1.2433 - learning_rate: 1.0000e-04
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 495ms/step - accuracy: 0.3349 - loss: 1.5165 - val_accuracy: 0.4933 - val_loss: 1.1581 - learning_rate: 1.0000e-04
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 511ms/step - accuracy: 0.3757 - loss: 1.4654 - val_accuracy: 0.5200 - val_loss: 1.0999 - learning_rate: 1.0000e-04
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 530ms/step - accuracy: 0.4510 - loss: 1.3360 - val_accuracy: 0.6400 - val_loss: 1.0452 - learning_rate: 1.0000e-04
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 530ms/step - accuracy: 0.5360 - loss: 1.2163 - val_accuracy: 0.6533 - val_loss: 0.9980 - learning_rate: 1.0000e-04
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 505ms/step - accuracy: 0.4852 - loss

In [16]:
print('phase2...finetuning')
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


phase2...finetuning


In [17]:
training_finetuning = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight,
    callbacks=[early_stop, lr_reduce]
)

Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 266s 11s/step - accuracy: 0.4962 - loss: 1.2130 - val_accuracy: 0.7200 - val_loss: 0.7956 - learning_rate: 1.0000e-05
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 541ms/step - accuracy: 0.5201 - loss: 1.1695 - val_accuracy: 0.7467 - val_loss: 0.7689 - learning_rate: 1.0000e-05
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 534ms/step - accuracy: 0.5702 - loss: 1.0484 - val_accuracy: 0.7733 - val_loss: 0.7439 - learning_rate: 1.0000e-05
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 518ms/step - accuracy: 0.6141 - loss: 1.0200 - val_accuracy: 0.8000 - val_loss: 0.7215 - learning_rate: 1.0000e-05
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 516ms/step - accuracy: 0.6252 - loss: 0.9042 - val_accuracy: 0.8000 - val_loss: 0.6952 - learning_rate: 1.0000e-05
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 504ms/step - accuracy: 0.6318 - loss: 0.9369 - val_accuracy: 0.8267 - val_loss: 0.6685 - learning_rate: 1.0000e-05
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 521ms/step - accuracy: 

In [19]:
best_train_acc = max(training_finetuning.history['accuracy'])
best_val_acc = max(training_finetuning.history['val_accuracy'])

In [20]:
print(f"Best Training Accuracy: {best_train_acc:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")

Best Training Accuracy: 0.7883
Best Validation Accuracy: 0.8533


In [22]:
print("Best Validation Accuracy:", max(training_finetuning.history['val_accuracy']))

Best Validation Accuracy: 0.8533333539962769
